In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

# ==========================================
# 1. LOAD LOCAL CSV DATASET
# ==========================================

df = pd.read_csv("IMDB_Dataset.csv")

print("Dataset loaded successfully!")
print(df.head())
print("Dataset shape:", df.shape)

# Remove missing values
df = df.dropna()

# Extract reviews and sentiments
X = df["review"].astype(str).values
y = df["sentiment"].values

# Convert positive/negative labels into 1/0
if y.dtype.kind in "OUS":
    y = np.array([
        1 if str(label).strip().lower() == "positive" else 0
        for label in y
    ])
else:
    y = y.astype("int32")

print("Total reviews:", len(X))
print("Positive reviews:", np.sum(y == 1))
print("Negative reviews:", np.sum(y == 0))


# ==========================================
# 2. SPLIT DATASET
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


# ==========================================
# 3. TOKENIZATION
# ==========================================

vocab_size = 10000
max_length = 200

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

# Learn vocabulary from training reviews only
tokenizer.fit_on_texts(X_train)

# Convert text into numerical sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


# ==========================================
# 4. PADDING SEQUENCES
# ==========================================

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("Training shape:", X_train_pad.shape)
print("Testing shape:", X_test_pad.shape)


# ==========================================
# 5. BUILD LSTM MODEL
# ==========================================

model = Sequential([

    # Word embedding
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        mask_zero=True
    ),

    # LSTM layer
    LSTM(64),

    # Dropout
    Dropout(0.5),

    # Binary classification
    Dense(1, activation="sigmoid")
])


# ==========================================
# 6. COMPILE MODEL
# ==========================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nMODEL SUMMARY")
model.summary()


# ==========================================
# 7. TRAIN MODEL
# ==========================================

history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)


# ==========================================
# 8. EVALUATE MODEL
# ==========================================

test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("\n========== MODEL RESULTS ==========")

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy * 100, "%")


# ==========================================
# 9. PLOT ACCURACY
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("LSTM Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")

plt.legend()
plt.grid()
plt.show()


# ==========================================
# 10. PLOT LOSS
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("LSTM Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")

plt.legend()
plt.grid()
plt.show()


# ==========================================
# 11. EXAMPLE PREDICTIONS
# ==========================================

predictions = model.predict(
    X_test_pad[:10],
    verbose=0
)

plt.figure(figsize=(12, 6))

for i in range(min(10, len(X_test_pad))):

    probability = predictions[i][0]

    predicted_label = (
        "Positive" if probability >= 0.5 else "Negative"
    )

    actual_label = (
        "Positive" if y_test[i] == 1 else "Negative"
    )

    print("\nReview:", X_test[i][:150])

    print("Actual:", actual_label)
    print("Predicted:", predicted_label)
    print("Confidence:", round(
        float(max(probability, 1 - probability)) * 100, 2
    ), "%")

    plt.subplot(2, 5, i + 1)

    plt.bar(
        ["Negative", "Positive"],
        [1 - probability, probability]
    )

    plt.title(predicted_label)
    plt.ylim(0, 1)

plt.tight_layout()
plt.show()


# ==========================================
# 12. CUSTOM REVIEW PREDICTION
# ==========================================

def predict_sentiment(review):

    sequence = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    if prediction >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print("Positive Probability:", round(float(prediction), 4))


predict_sentiment(
    "This movie was amazing and the acting was excellent"
)

predict_sentiment(
    "This movie was boring and completely disappointing"
)


# ==========================================
# 13. SAVE MODEL
# ==========================================

model.save("imdb_lstm_model.keras")

print("\nModel saved successfully!")
print("Practical completed!")


Dataset loaded successfully!
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Dataset shape: (50000, 2)
Total reviews: 50000
Positive reviews: 25000
Negative reviews: 25000
Training samples: 40000
Testing samples: 10000
Training shape: (40000, 200)
Testing shape: (10000, 200)

MODEL SUMMARY


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
360/500 ━━━━━━━━━━━━━━━━━━━━ 25s 184ms/step - accuracy: 0.6949 - loss: 0.5714